## 02 — Data Assessment

> **Read-only audit of the raw NBA game-level dataset.**  
> No data is modified, imputed, or removed in this notebook.  
> Output: a structured quality report. Remediation actions are deferred to `03_data_cleaning.ipynb`.

---

**Pipeline**

1. Setup — imports, interactive tables, visualization defaults, and quality thresholds
2. Load Raw Data — dataset version resolution, metadata loading, and SHA-256 verification
3. Structural Overview — schema contract validation, shape checks, preview, and sampling
4. Data Quality Assessment — completeness, uniqueness, validity, and consistency diagnostics
5. Assessment Report — summary status table and per-column completeness scores
6. Conclusion — dynamic interpretation of quality findings and cleaning priorities

### 1. Setup

* **What:** Imports audit utilities, configures the project root, initializes interactive table rendering, sets plotting defaults, and defines quality-control thresholds.
* **Why:** To prepare a consistent read-only assessment environment for inspecting the raw dataset without introducing transformations.
* **Reasoning/Choices:** `itables` improves wide-schema exploration, while explicit thresholds such as `NULL_THRESHOLD` and `TOLERANCE` make quality decisions reproducible and transparent.

In [ ]:
import hashlib
import json
import logging
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().resolve().parent.parent
sys.path.append(str(ROOT))

import itables.options as opt
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from itables import init_notebook_mode, show

from src.loader import load_config
# Load configuration settings from config.toml
config = load_config()

# Clean logging configuration tailored for Jupyter notebooks and scripts
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

# Initialize notebook mode for interactive display
init_notebook_mode(all_interactive=True)
opt.warn_on_undocumented_option = False

opt.maxBytes   = 0
opt.pageLength = 10
opt.lengthMenu = [10, 25, 50, 100]
opt.scrollX    = True
opt.classes    = "display nowrap compact"
opt.style      = "width:100%; font-size:13px;"

# Set seaborn theme and figure parameters
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120})

NOTEBOOK_VERSION = "1.1.0"
NULL_THRESHOLD   = 0.30   # critical completeness threshold
TOLERANCE        = 0.005  # rounding tolerance for pct cross-checks

log.info(f"Assessment notebook v{NOTEBOOK_VERSION} — initialized")

### 2. Load Raw Data

* **What:** Resolves the raw dataset path from configuration, loads ingestion metadata, verifies the SHA-256 checksum, and reads the CSV into a Polars DataFrame.
* **Why:** To guarantee that the assessment is performed on the exact raw artifact produced by `01_data_ingestion.ipynb`.
* **Reasoning/Choices:** Hash verification prevents silent file drift between ingestion and assessment. Reading with an extended schema inference window improves type detection while preserving the notebook as a non-mutating audit stage.

In [ ]:
# Retrieve settings from config
RAW_DATA_DIR = ROOT / config["data"]["raw_data_dir"]

start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

start_season = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season   = f"{end_year}-{str(end_year + 1)[-2:]}"

# Define dataset version
DATASET_VERSION = f"{start_season}_{end_season}"

RAW_FILE = RAW_DATA_DIR / f"nba_stats_{DATASET_VERSION}.csv"

log.info(f"Source file     : {RAW_FILE}")
log.info(f"Dataset version : {DATASET_VERSION}")

METADATA_FILE = RAW_DATA_DIR / f"metadata_{DATASET_VERSION}.json"

with open(METADATA_FILE, "r") as f:
    metadata = json.load(f)

# Extract the expected hash that was saved during Data Ingestion
EXPECTED_SHA256 = metadata["sha256"]

# Calculate the current hash of the raw file
current_sha256 = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()

# Compare the current hash with the expected hash
if current_sha256 != EXPECTED_SHA256:
    log.error(
        f"INTEGRITY ERROR! Expected hash: {EXPECTED_SHA256}, but found {current_sha256}"
    ) 

df = pl.read_csv(RAW_FILE, infer_schema_length=10_000, try_parse_dates=False)

n_rows, n_cols = df.shape

log.info(f"Loaded {n_rows:,} rows x {n_cols} columns")

### 3. Structural Overview

* **What:** Compares the raw dataset against the expected schema, reports shape information, highlights missing or unexpected columns, and displays representative records.
* **Why:** To verify that the raw data contract is intact before evaluating deeper quality dimensions.
* **Reasoning/Choices:** Separating schema validation from data-quality checks makes it easier to distinguish structural ingestion issues from legitimate missingness, duplicates, or statistical anomalies.

In [ ]:
# Contractual schema: column → expected Polars dtype
EXPECTED_SCHEMA: dict[str, pl.DataType] = {
    # Identifiers (traditional split)
    "gameId":                          pl.String,
    "teamId":                          pl.String,
    "teamCity":                        pl.String,
    "teamName":                        pl.String,
    "teamTricode":                     pl.String,
    "teamSlug":                        pl.String,
    # Traditional stats (starter / bench split)
    "minutes":                         pl.String,
    "fieldGoalsMade":                  pl.Int64,
    "fieldGoalsAttempted":             pl.Int64,
    "fieldGoalsPercentage":            pl.Float64,
    "threePointersMade":               pl.Int64,
    "threePointersAttempted":          pl.Int64,
    "threePointersPercentage":         pl.Float64,
    "freeThrowsMade":                  pl.Int64,
    "freeThrowsAttempted":             pl.Int64,
    "freeThrowsPercentage":            pl.Float64,
    "reboundsOffensive":               pl.Int64,
    "reboundsDefensive":               pl.Int64,
    "reboundsTotal":                   pl.Int64,
    "assists":                         pl.Int64,
    "steals":                          pl.Int64,
    "blocks":                          pl.Int64,
    "turnovers":                       pl.Int64,
    "foulsPersonal":                   pl.Int64,
    "points":                          pl.Int64,
    "startersBench":                   pl.String,
    # Advanced stats (team-level aggregates, duplicated from join)
    "teamCity_adv":                    pl.String,
    "teamName_adv":                    pl.String,
    "teamTricode_adv":                 pl.String,
    "teamSlug_adv":                    pl.String,
    "minutes_adv":                     pl.String,
    "estimatedOffensiveRating":        pl.Float64,
    "offensiveRating":                 pl.Float64,
    "estimatedDefensiveRating":        pl.Float64,
    "defensiveRating":                 pl.Float64,
    "estimatedNetRating":              pl.Float64,
    "netRating":                       pl.Float64,
    "assistPercentage":                pl.Float64,
    "assistToTurnover":                pl.Float64,
    "assistRatio":                     pl.Float64,
    "offensiveReboundPercentage":      pl.Float64,
    "defensiveReboundPercentage":      pl.Float64,
    "reboundPercentage":               pl.Float64,
    "estimatedTeamTurnoverPercentage": pl.Float64,
    "turnoverRatio":                   pl.Float64,
    "effectiveFieldGoalPercentage":    pl.Float64,
    "trueShootingPercentage":          pl.Float64,
    "usagePercentage":                 pl.Float64,
    "estimatedUsagePercentage":        pl.Float64,
    "estimatedPace":                   pl.Float64,
    "pace":                            pl.Float64,
    "pacePer40":                       pl.Float64,
    "possessions":                     pl.Float64,
    "PIE":                             pl.Float64,
    # LeagueGameFinder columns (joined; partially redundant with above)
    "SEASON_ID":                       pl.String,
    "TEAM_ABBREVIATION":               pl.String,
    "TEAM_NAME":                       pl.String,
    "GAME_DATE":                       pl.String,
    "MATCHUP":                         pl.String,
    "WL":                              pl.String,
    "MIN":                             pl.Int64,
    "PTS":                             pl.Int64,
    "FGM":                             pl.Int64,
    "FGA":                             pl.Int64,
    "FG_PCT":                          pl.Float64,
    "FG3M":                            pl.Int64,
    "FG3A":                            pl.Int64,
    "FG3_PCT":                         pl.Float64,
    "FTM":                             pl.Int64,
    "FTA":                             pl.Int64,
    "FT_PCT":                          pl.Float64,
    "OREB":                            pl.Int64,
    "DREB":                            pl.Int64,
    "REB":                             pl.Int64,
    "AST":                             pl.Int64,
    "STL":                             pl.Int64,
    "BLK":                             pl.Int64,
    "TOV":                             pl.Int64,
    "PF":                              pl.Int64,
    "PLUS_MINUS":                      pl.Float64,
    "SEASON":                          pl.String,
}

actual_cols   = set(df.columns)
expected_cols = set(EXPECTED_SCHEMA.keys())

missing_cols    = expected_cols - actual_cols
unexpected_cols = actual_cols  - expected_cols

print(f"Shape            : {n_rows:,} rows x {n_cols} columns")
print(f"Expected columns : {len(expected_cols)}")
print(f"Actual columns   : {len(actual_cols)}")
print(f"Missing columns  : {missing_cols  or 'none'}")
print(f"Unexpected cols  : {unexpected_cols or 'none'}\n")

schema_rows = []

# Check schema consistency
for col in df.columns:
    actual_dtype   = str(df[col].dtype)
    expected_dtype = str(EXPECTED_SCHEMA.get(col, "NOT IN CONTRACT"))
    match_flag     = "OK" if actual_dtype == expected_dtype else "MISMATCH"
    schema_rows.append({
        "column":         col,
        "actual_dtype":   actual_dtype,
        "expected_dtype": expected_dtype,
        "match":          match_flag,
    })

# Create DataFrame of schema mismatches
df_schema = pl.DataFrame(schema_rows)
mismatches = df_schema.filter(pl.col("match") == "MISMATCH")
log.info(f"Schema mismatches: {mismatches.height}")

# Display schema mismatches
show(df_schema)

In [ ]:
# Retrieve assessment settings from config
preview_rows = config["assessment"]["preview_rows"]
sample_rows  = config["assessment"]["sample_rows"]
random_seed  = config["assessment"]["random_seed"]

# Display preview and sample rows
print("=== First 3 rows ===")
display(df.head(preview_rows))

print("\n=== Random 3-row sample (seed=42) ===")
display(df.sample(sample_rows, seed=random_seed))

### 4. Data Quality Assessment

* **What:** Audits the dataset across four core dimensions: completeness, uniqueness, validity, and consistency.
* **Why:** To identify actionable data-quality issues before any cleaning rule is applied.
* **Reasoning/Choices:** The assessment remains strictly diagnostic. By deferring remediation to the cleaning notebook, each data-quality decision remains traceable from finding to transformation.

#### 4.1 Completeness

* **What:** Computes null counts and null percentages for every column, identifies columns above the critical missingness threshold, and checks for fully empty records.
* **Why:** To determine whether missing data threatens downstream feature engineering, target construction, or model training.
* **Reasoning/Choices:** Reporting both counts and percentages supports comparison across columns with different practical importance. High-null columns are flagged rather than removed because this notebook is intentionally read-only.

In [ ]:
# Calculate null counts for each column
null_counts = (
    df
    .null_count()
    .unpivot(variable_name="column", value_name="null_count")
    .with_columns(
        (pl.col("null_count") / n_rows).alias("null_pct")
    )
    .sort("null_pct", descending=True)
)

# Filter columns with nulls above threshold
critical_cols = null_counts.filter(pl.col("null_pct") > NULL_THRESHOLD)

# Count columns with any nulls
n_cols_any_null = null_counts.filter(pl.col("null_count") > 0).height
log.info(f"Columns with any nulls        : {n_cols_any_null}")

# Display critical columns
log.info(f"Critical columns (>{NULL_THRESHOLD:.0%} null)  : {critical_cols.height}")

if critical_cols.height > 0:
    log.warning(f"CRITICAL: {critical_cols['column'].to_list()}")

# Display columns with nulls only
print("\n--- Null counts (columns with nulls only) ---")
display(null_counts.filter(pl.col("null_count") > 0))

# Filter completely empty records
empty_records = df.filter(
    pl.all_horizontal([pl.col(c).is_null() for c in df.columns])
)
log.info(f"Completely empty records        : {empty_records.height}")

#### 4.2 Uniqueness

* **What:** Measures exact duplicate rows, duplicate business keys, and unexpected starter/bench split patterns at the game-team level.
* **Why:** To detect repeated or structurally ambiguous records that could bias model training or inflate sample counts.
* **Reasoning/Choices:** The audit distinguishes true duplicates from expected NBA API structure, where each team-game can appear separately for Starters and Bench. This distinction informs the deduplication logic in the cleaning notebook.

In [ ]:
# Calculate exact duplicate rows
n_exact_dupes = df.height - df.unique().height
log.info(f"Exact duplicate rows: {n_exact_dupes:,}")

# Display sample of exact duplicates
if n_exact_dupes > 0:
    dupes = df.filter(df.is_duplicated())
    log.info(f"Sample of exact duplicates ({min(5, dupes.height)} of {dupes.height}):")
    display(dupes.head(5))
else:
    log.info("No exact duplicate rows found.")

In [ ]:
# Business key: (gameId, teamId, startersBench) — must be unique
BK_COLS = ["gameId", "teamId", "startersBench"]

# Calculate duplicate (gameId, teamId, startersBench) combos
bk_dupes = (
    df
    .group_by(BK_COLS)
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") > 1)
    .sort("count", descending=True)
)
log.info(f"Duplicate (gameId, teamId, startersBench) combos: {bk_dupes.height:,}")

if bk_dupes.height > 0:
    print(bk_dupes.head(10))

# Each (gameId, teamId) pair must have exactly 2 rows: Starters + Bench
lgf_anomalies = (
    df
    .group_by(["gameId", "teamId"])
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") != 2)
    .sort("count", descending=True)
)
log.info(
    f"(gameId, teamId) combos with count != 2 "
    f"(expected: Starters + Bench): {lgf_anomalies.height:,}"
)
if lgf_anomalies.height > 0:
    print(lgf_anomalies.head(10))

In [ ]:
# Near-duplicates: same game + team but unexpected split count
near_dup_candidates = (
    df
    .group_by(["gameId", "teamId"])
    .agg(
        pl.col("startersBench").n_unique().alias("n_splits"),
        pl.len().alias("n_rows"),
    )
    .filter(pl.col("n_splits") > 2)
    .sort("n_rows", descending=True)
)
log.info(
    f"Games with more than 2 starter/bench splits "
    f"(potential near-duplicates): {near_dup_candidates.height:,}"
)
if near_dup_candidates.height > 0:
    display(near_dup_candidates.head(10))
else:
    log.info("No near-duplicate patterns detected.")

#### 4.3 Validity

* **What:** Checks categorical domains, numeric ranges, percentage bounds, date formats, and configured date coverage.
* **Why:** To verify that raw observations obey the physical and semantic constraints expected from NBA box-score data.
* **Reasoning/Choices:** Domain validation separates corrupted records from legitimate outliers. Extreme but valid basketball performances should remain available for modeling, while impossible values should be flagged for cleaning.

In [ ]:
# WL domain: {W, L}
wl_values = df["WL"].drop_nulls().unique().sort().to_list()
log.info(f"WL distinct values      : {wl_values}")
invalid_wl = df.filter(
    ~pl.col("WL").is_in(["W", "L"]) & pl.col("WL").is_not_null()
)
log.info(f"Invalid WL records      : {invalid_wl.height}")

# startersBench domain: {Starters, Bench}
sb_values = df["startersBench"].drop_nulls().unique().sort().to_list()
log.info(f"startersBench values    : {sb_values}")
# Select records with invalid startersBench values
invalid_sb = df.filter(
    ~pl.col("startersBench").is_in(["Starters", "Bench"])
    & pl.col("startersBench").is_not_null()
)
log.info(f"Invalid startersBench   : {invalid_sb.height}")

# MATCHUP: must contain " vs. " (home game) or " @ " (away game)
invalid_matchup = df.filter(
    ~pl.col("MATCHUP").str.contains(r" vs\. | @ ")
    & pl.col("MATCHUP").is_not_null()
)
log.info(f"Invalid MATCHUP format  : {invalid_matchup.height}")

# SEASON: must match YYYY-YY pattern
invalid_season = df.filter(
    ~pl.col("SEASON").str.contains(r"^\d{4}-\d{2}$")
    & pl.col("SEASON").is_not_null()
)
log.info(f"Invalid SEASON format   : {invalid_season.height}")

if invalid_wl.height > 0:
    print("Invalid WL sample:"); print(invalid_wl.head(5))
if invalid_sb.height > 0:
    print("Invalid startersBench sample:"); print(invalid_sb.head(5))

In [ ]:
# Columns that must be >= 0
NON_NEGATIVE_COLS = [
    "fieldGoalsMade", "fieldGoalsAttempted",
    "threePointersMade", "threePointersAttempted",
    "freeThrowsMade", "freeThrowsAttempted",
    "reboundsOffensive", "reboundsDefensive", "reboundsTotal",
    "assists", "steals", "blocks", "turnovers", "foulsPersonal", "points",
    "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PTS", "MIN",
]

range_issues: dict[str, int] = {}
for col in NON_NEGATIVE_COLS:
    if col not in df.columns:
        continue
    n_neg = df.filter(
        pl.col(col).cast(pl.Float64, strict=False) < 0
    ).height
    if n_neg > 0:
        range_issues[col] = n_neg
        log.warning(f"Column '{col}': {n_neg} negative value(s)")

if not range_issues:
    log.info("No negative values found in non-negative columns")
else:
    print("Range violations:", range_issues)

In [ ]:
# Percentage columns must be in [0, 1]
PCT_COLS = [
    ("fieldGoalsPercentage",         0.0, 1.0),
    ("threePointersPercentage",      0.0, 1.0),
    ("freeThrowsPercentage",         0.0, 1.0),
    ("FG_PCT",                       0.0, 1.0),
    ("FG3_PCT",                      0.0, 1.0),
    ("FT_PCT",                       0.0, 1.0),
    ("effectiveFieldGoalPercentage", 0.0, 1.0),
    ("trueShootingPercentage",       0.0, 1.0),
]

pct_issues: dict[str, int] = {}
for col, lo, hi in PCT_COLS:
    if col not in df.columns:
        continue
    n_bad = df.filter(
        pl.col(col).cast(pl.Float64, strict=False).is_not_null()
        & (
            (pl.col(col).cast(pl.Float64, strict=False) < lo)
            | (pl.col(col).cast(pl.Float64, strict=False) > hi)
        )
    ).height
    if n_bad > 0:
        pct_issues[col] = n_bad
        log.warning(f"Percentage column '{col}': {n_bad} value(s) outside [{lo}, {hi}]")

if not pct_issues:
    log.info("All percentage columns are within [0, 1]")
else:
    print("Percentage violations:", pct_issues)

In [ ]:
invalid_date_format = df.filter(
    ~pl.col("GAME_DATE").str.contains(r"^\d{4}-\d{2}-\d{2}$")
    & pl.col("GAME_DATE").is_not_null()
)
log.info(f"Records with invalid GAME_DATE format : {invalid_date_format.height}")

# Convert GAME_DATE to datetime and filter out invalid dates
df_dates = df.with_columns(
    pl.col("GAME_DATE")
    .str.strptime(pl.Date, "%Y-%m-%d", strict=False)
    .alias("_date")
)
min_date = df_dates["_date"].min()
max_date = df_dates["_date"].max()
log.info(f"GAME_DATE observed range: {min_date} -> {max_date}")

# Get earliest and latest dates from config
earliest_date = config["assessment"]["earliest_date"]
latest_date = config["assessment"]["latest_date"]

# Convert config dates to datetime objects
earliest_date = datetime.strptime(earliest_date, "%Y-%m-%d").date()
latest_date = datetime.strptime(latest_date, "%Y-%m-%d").date()

# Filter records with GAME_DATE outside the specified range
out_of_range = df_dates.filter(
    (pl.col("_date") < earliest_date) | (pl.col("_date") > latest_date)
)
log.info(
    f"Records with GAME_DATE outside [{earliest_date}, {latest_date}]: {out_of_range.height}"
)
if out_of_range.height > 0:
    display(out_of_range.select(["gameId", "GAME_DATE", "SEASON"]).head(10))

#### 4.4 Consistency

* **What:** Recomputes arithmetic relationships among made/attempted statistics, rebound totals, stored percentages, and season identifiers.
* **Why:** To detect internal contradictions even when individual values appear valid in isolation.
* **Reasoning/Choices:** Consistency checks use domain-specific basketball invariants and a small percentage tolerance to account for rounding. These checks provide targeted evidence for cleaning decisions without altering the raw table.

In [ ]:
# Cast column to float64 and return expression
def _cast(col: str) -> pl.Expr:
    return pl.col(col).cast(pl.Float64, strict=False)

# Dictionary to store consistency flags and counts
consistency_flags: dict[str, int] = {}

checks = [
    ("fieldGoalsMade > fieldGoalsAttempted",
     _cast("fieldGoalsMade") > _cast("fieldGoalsAttempted")),
    ("threePointersMade > threePointersAttempted",
     _cast("threePointersMade") > _cast("threePointersAttempted")),
    ("freeThrowsMade > freeThrowsAttempted",
     _cast("freeThrowsMade") > _cast("freeThrowsAttempted")),
    ("reboundsOff + reboundsDef != reboundsTotal",
     ((_cast("reboundsOffensive") + _cast("reboundsDefensive")) != _cast("reboundsTotal"))
     & pl.col("reboundsTotal").is_not_null()),
    ("FGM > FGA",  _cast("FGM")  > _cast("FGA")),
    ("FG3M > FG3A", _cast("FG3M") > _cast("FG3A")),
    ("FTM > FTA",  _cast("FTM")  > _cast("FTA")),
    ("OREB + DREB != REB",
     ((_cast("OREB") + _cast("DREB")) != _cast("REB")) & pl.col("REB").is_not_null()),
]

# Iterate through checks and count inconsistencies
for label, expr in checks:
    n = df.filter(expr).height
    consistency_flags[label] = n

print("\nConsistency cross-checks:")
for label, n in consistency_flags.items():
    status = "OK" if n == 0 else f"FAIL  ({n:,} records)"
    print(f"  {label:<55}  {status}")

In [ ]:
# Sanity check: verify stored percentages against recomputed values
pct_cross_checks = [
    ("FGM",  "FGA",  "FG_PCT"),
    ("FG3M", "FG3A", "FG3_PCT"),
    ("FTM",  "FTA",  "FT_PCT"),
]

n_mismatch: dict[str, int] = {}
for made, att, pct in pct_cross_checks:
    n = (
        df
        .filter(pl.col(att).cast(pl.Float64, strict=False) > 0)
        .with_columns(
            (
                pl.col(made).cast(pl.Float64, strict=False)
                / pl.col(att).cast(pl.Float64, strict=False)
            ).alias("_calc")
        )
        .filter(
            (pl.col("_calc") - pl.col(pct).cast(pl.Float64, strict=False)).abs()
            > TOLERANCE
        )
        .height
    )
    n_mismatch[pct] = n
    status = "OK" if n == 0 else f"MISMATCH  ({n:,} records)"
    log.info(f"{pct} recompute check: {status}")

print("\nStored vs. recomputed percentage checks (tolerance =", TOLERANCE, "):")
for pct, n in n_mismatch.items():
    print(f"  {pct:<12}  {'OK' if n == 0 else f'FAIL ({n:,})'}")

In [ ]:
# SEASON_ID encodes the season start year in digits 1-4 (NBA format: "2YYYY...")
# SEASON column: "YYYY-YY"
df_season_check = df.with_columns([
    # Cast to pl.String first so we can use string slicing
    pl.col("SEASON_ID").cast(pl.String).str.slice(1, 4).cast(pl.Int64, strict=False).alias("_sid_year"),
    pl.col("SEASON").cast(pl.String).str.slice(0, 4).cast(pl.Int64, strict=False).alias("_s_year"),
]).filter(
    pl.col("SEASON_ID").is_not_null() & pl.col("SEASON").is_not_null()
)

season_mismatch = df_season_check.filter(
    pl.col("_sid_year") != pl.col("_s_year")
)
log.info(f"SEASON_ID / SEASON year mismatch   : {season_mismatch.height:,} records")

if season_mismatch.height > 0:
    print(season_mismatch.select(["gameId", "SEASON_ID", "SEASON"]).head(10))
else:
    print("SEASON_ID and SEASON are consistent across all records.")

### 5. Assessment Report

* **What:** Aggregates all quality checks into a concise report by dimension and computes per-column completeness scores.
* **Why:** To provide a structured handoff from assessment to cleaning, with clear priorities and severity labels.
* **Reasoning/Choices:** Summarizing by dimension keeps the report readable while preserving detail in the preceding diagnostic cells. The resulting table acts as the decision log for `03_data_cleaning.ipynb`.

In [ ]:
# Calculate total validity issues
total_validity_issues = (
    invalid_wl.height
    + invalid_sb.height
    + invalid_matchup.height
    + invalid_season.height
    + sum(range_issues.values())
    + sum(pct_issues.values())
    + invalid_date_format.height
    + out_of_range.height
)

# Calculate total consistency issues
total_consistency_issues = (
    sum(consistency_flags.values())
    + sum(n_mismatch.values())
    + season_mismatch.height
)

report_rows = [
    {
        "dimension":   "Completeness",
        "checks_run":  "null counts per column + empty records",
        "n_issues":    null_counts.filter(pl.col("null_count") > 0).height,
        "n_critical":  critical_cols.height,
        "status":      "FAIL" if critical_cols.height > 0 else "PASS",
    },
    {
        "dimension":   "Uniqueness",
        "checks_run":  "exact duplicates + business key + near-duplicates",
        "n_issues":    n_exact_dupes + bk_dupes.height,
        "n_critical":  n_exact_dupes,
        "status":      "FAIL" if n_exact_dupes > 0 else ("WARN" if bk_dupes.height > 0 else "PASS"),
    },
    {
        "dimension":   "Validity",
        "checks_run":  "domain + range + percentages + dates",
        "n_issues":    total_validity_issues,
        "n_critical":  invalid_wl.height + sum(range_issues.values()),
        "status":      "FAIL" if (invalid_wl.height + sum(range_issues.values())) > 0 else ("WARN" if total_validity_issues > 0 else "PASS"),
    },
    {
        "dimension":   "Consistency",
        "checks_run":  "arithmetic cross-checks + stored vs. computed + seasonal IDs",
        "n_issues":    total_consistency_issues,
        "n_critical":  sum(v for v in consistency_flags.values() if v > 0),
        "status":      "FAIL" if any(v > 0 for v in consistency_flags.values()) else ("WARN" if total_consistency_issues > 0 else "PASS"),
    },
]

df_report = pl.DataFrame(report_rows)
print("=== Assessment Report — Summary by Dimension ===\n")
display(df_report)

In [ ]:
# Calculate per-column completeness score
df_col_scores = (
    null_counts
    .with_columns(
        (1.0 - pl.col("null_pct")).round(4).alias("completeness_score")
    )
    .select(["column", "null_count", "null_pct", "completeness_score"])
)

print("=== Per-column Completeness Score ===\n")
show(df_col_scores)

### 6. Conclusion

* **What:** Interprets the quality report across schema integrity, completeness, uniqueness, validity, and consistency checks.
* **Why:** To translate diagnostics into cleaning priorities without modifying the raw dataset inside this read-only audit notebook.
* **Reasoning/Choices:** The conclusion should be driven by the computed assessment tables, so remediation decisions remain tied to measured quality issues rather than assumptions about the data.

In [ ]:
# Cell: Dynamic Conclusion Generation
from IPython.display import Markdown, display

schema_mismatch_count = mismatches.height if "mismatches" in globals() else 0
critical_null_count = critical_cols.height if "critical_cols" in globals() else 0
exact_duplicate_count = n_exact_dupes if "n_exact_dupes" in globals() else 0
business_key_duplicate_count = bk_dupes.height if "bk_dupes" in globals() else 0
validity_issue_count = total_validity_issues if "total_validity_issues" in globals() else 0
consistency_issue_count = total_consistency_issues if "total_consistency_issues" in globals() else 0
report_statuses = df_report["status"].to_list() if "df_report" in globals() else []
overall_status = "PASS" if report_statuses and all(status == "PASS" for status in report_statuses) else "REVIEW REQUIRED"

conclusion = f"""
This section dynamically summarizes the data-quality assessment based on the current notebook execution.

#### 6.1 Structural Integrity
The raw dataset contains **{n_rows:,} rows** and **{n_cols} columns**. Schema validation identified **{schema_mismatch_count} dtype mismatch(es)** against the expected raw-data contract.

#### 6.2 Quality Dimensions
Completeness checks found **{critical_null_count} critical high-null column(s)**, uniqueness checks found **{exact_duplicate_count:,} exact duplicate row(s)** and **{business_key_duplicate_count:,} duplicate business-key combination(s)**, validity checks found **{validity_issue_count:,} issue(s)**, and consistency checks found **{consistency_issue_count:,} issue(s)**.

#### 6.3 Assessment Status
The overall assessment status is **{overall_status}**. Any `FAIL` or `WARN` dimension should be translated into explicit cleaning rules rather than handled implicitly during feature engineering or model training.

#### 6.4 Next Step
The findings in this notebook define the remediation plan for `03_data_cleaning.ipynb`, where duplicates, invalid records, schema types, categorical domains, and mandatory missing values are handled in a controlled and traceable way.
"""

display(Markdown(conclusion))